[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-repo/rl-textbook/blob/main/notebooks/ch11_lm_post_training.ipynb)

<div style="background: linear-gradient(135deg, #1B474D 0%, #20808D 100%); padding: 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: white; margin: 0; font-size: 2em;">Chapter 11 — Language Models & Post-Training</h1>
  <p style="color: #BCE2E7; margin-top: 10px; font-size: 1.1em;">
    Understanding autoregressive generation, sampling strategies, and Supervised Fine-Tuning (SFT).
    We load GPT-2 small, explore decoding, fine-tune on a tiny instruction dataset, and measure perplexity.
  </p>
</div>

**Learning objectives:**
- Load and run GPT-2 small from HuggingFace Transformers
- Compare temperature, top-k, and top-p sampling strategies
- Fine-tune GPT-2 via SFT on 100 instruction examples
- Measure perplexity before and after fine-tuning

<div style="background: #1C1B19; border-left: 4px solid #20808D; padding: 15px; border-radius: 6px; margin: 10px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">Environment Setup</h2>
  <p style="color: #CDCCCA; margin-top: 8px;">Requires a T4 GPU on Colab for comfortable runtime (~12 min). CPU will work but fine-tuning will take ~20–30 min. Set runtime type to GPU: Runtime → Change runtime type → T4 GPU.</p>
</div>

In [ ]:
!pip install -q transformers accelerate datasets trl torch matplotlib numpy tqdm

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer, get_scheduler
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from tqdm.auto import tqdm
import warnings, math, time
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

plt.rcParams.update({
    'figure.facecolor': '#F7F6F2', 'axes.facecolor': '#F9F8F5',
    'axes.edgecolor': '#D4D1CA', 'axes.labelcolor': '#28251D',
    'text.color': '#28251D', 'xtick.color': '#7A7974',
    'ytick.color': '#7A7974', 'grid.color': '#D4D1CA',
    'font.family': 'sans-serif',
})

<div style="background: #1C1B19; border-left: 4px solid #A84B2F; padding: 15px; border-radius: 6px; margin: 15px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">§11.1 Autoregressive Generation with GPT-2</h2>
</div>

GPT-2 is a **decoder-only transformer** trained to predict the next token given all previous tokens:

\[ P(x_1, \ldots, x_T) = \prod_{t=1}^{T} P(x_t \mid x_1, \ldots, x_{t-1}) \]

At each step the model outputs a **logit vector** over the vocabulary. We apply softmax to get probabilities, then **sample** (or argmax) to pick the next token.

GPT-2 small has **124M parameters** — small enough to fit on a Colab T4 with headroom for fine-tuning.

In [ ]:
# Load GPT-2 small (~500 MB, ~30s)
MODEL_NAME = 'gpt2'  # 124M params
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'GPT-2 small: {n_params/1e6:.1f}M parameters')
print(f'Vocabulary size: {tokenizer.vocab_size:,}')
print(f'Max context length: {model.config.n_ctx} tokens')

In [ ]:
# Visualise next-token probability distribution
prompt = "The key to aligning language models with human values is"
inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits[0, -1, :]  # last position
    probs = F.softmax(logits, dim=-1)

topk = 15
top_probs, top_ids = probs.topk(topk)
top_tokens = [tokenizer.decode(tid) for tid in top_ids.cpu()]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(range(topk), top_probs.cpu().numpy()[::-1], color='#20808D')
ax.set_yticks(range(topk))
ax.set_yticklabels([repr(t) for t in top_tokens[::-1]], fontsize=9, fontfamily='monospace')
ax.set_xlabel('Next-Token Probability')
ax.set_title(f'§11.1 — Top-{topk} Next Token Probabilities\nPrompt: "{prompt}"', fontweight='bold')
plt.tight_layout()
plt.savefig('ch11_fig1_next_token_probs.png', dpi=120, bbox_inches='tight')
plt.show()

<div style="background: #1C1B19; border-left: 4px solid #A84B2F; padding: 15px; border-radius: 6px; margin: 15px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">§11.2 Sampling Strategies: Temperature, Top-k, Top-p</h2>
</div>

Three key knobs control how we sample from the model's next-token distribution:

- **Temperature** \(T\): scale logits by \(1/T\) before softmax. \(T < 1\) sharpens; \(T > 1\) flattens.
- **Top-k**: restrict sampling to the \(k\) most probable tokens.
- **Top-p (nucleus)**: restrict to the smallest set of tokens whose cumulative probability ≥ \(p\).

In [ ]:
def generate_with_strategy(prompt, strategy='greedy', temperature=1.0,
                            top_k=0, top_p=1.0, max_new_tokens=40):
    inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        if strategy == 'greedy':
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        elif strategy == 'temperature':
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                                  temperature=temperature)
        elif strategy == 'topk':
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                                  top_k=top_k)
        elif strategy == 'topp':
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                                  top_p=top_p, top_k=0)
    return tokenizer.decode(out[0], skip_special_tokens=True)

PROMPT = "Once upon a time, a curious researcher discovered"

configs = [
    ('Greedy (T→0)',     'greedy',      {}),
    ('Temperature=0.5',  'temperature', {'temperature': 0.5}),
    ('Temperature=1.0',  'temperature', {'temperature': 1.0}),
    ('Temperature=1.5',  'temperature', {'temperature': 1.5}),
    ('Top-k=10',         'topk',        {'top_k': 10}),
    ('Top-p=0.9 (nucleus)', 'topp',     {'top_p': 0.9}),
]

print(f'Prompt: "{PROMPT}"\n')
for name, strat, kwargs in configs:
    text = generate_with_strategy(PROMPT, strategy=strat, **kwargs)
    generated = text[len(PROMPT):].strip()
    print(f'[{name}]\n  {generated[:120]}\n')

In [ ]:
# Visualise how temperature reshapes the distribution
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits[0, -1, :]

top_ids_base = logits.topk(30).indices
temperatures = [0.3, 0.7, 1.0, 1.5, 2.0]
colors = ['#1B474D', '#20808D', '#BCE2E7', '#FFC553', '#A84B2F']

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(30)
for T, col in zip(temperatures, colors):
    p = F.softmax(logits[top_ids_base] / T, dim=-1).cpu().numpy()
    ax.plot(x, p, marker='.', label=f'T={T}', color=col, linewidth=1.8)

ax.set_xlabel('Token rank (top 30 by base logit)')
ax.set_ylabel('Probability after temperature scaling')
ax.set_title('§11.2 — Temperature Effect on Next-Token Distribution', fontweight='bold')
ax.legend(title='Temperature')
ax.grid(alpha=0.4)
plt.tight_layout()
plt.savefig('ch11_fig2_sampling_strategies.png', dpi=120, bbox_inches='tight')
plt.show()

<div style="background: #1C1B19; border-left: 4px solid #A84B2F; padding: 15px; border-radius: 6px; margin: 15px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">§11.3 Supervised Fine-Tuning (SFT) on Instruction Data</h2>
</div>

**SFT** is the first stage of post-training: we fine-tune the base LM on a dataset of (instruction, response) pairs using standard next-token prediction loss. This adapts the model's behaviour toward following instructions.

We use 100 examples from the **Databricks Dolly-15k** dataset. The fine-tuning loss is:

\[ \mathcal{L}_{\text{SFT}} = -\frac{1}{|\mathcal{D}|}\sum_{(x,y) \in \mathcal{D}} \sum_{t=1}^{|y|} \log P_\theta(y_t \mid x, y_{<t}) \]

Only the **response tokens** contribute to the loss (the instruction is used as context only).

In [ ]:
from datasets import load_dataset

# Load tiny subset of Dolly (~30s)
ds = load_dataset('databricks/databricks-dolly-15k', split='train')
ds = ds.filter(lambda x: x['category'] == 'open_qa' and len(x['response']) < 200)
ds = ds.select(range(min(100, len(ds))))
print(f'Fine-tuning set: {len(ds)} examples')
print('\nExample:')
print('  Instruction:', ds[0]['instruction'][:80])
print('  Response   :', ds[0]['response'][:80])

In [ ]:
class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        # Format: ### Instruction:\n{inst}\n\n### Response:\n{resp}
        prompt = f"### Instruction:\n{row['instruction']}\n\n### Response:\n"
        full_text = prompt + row['response'] + tokenizer.eos_token

        enc = self.tokenizer(
            full_text, truncation=True, max_length=self.max_length,
            padding='max_length', return_tensors='pt'
        )
        input_ids = enc.input_ids.squeeze()
        attention_mask = enc.attention_mask.squeeze()

        # Mask instruction tokens in labels (train only on response)
        prompt_enc = self.tokenizer(prompt, return_tensors='pt')
        prompt_len = prompt_enc.input_ids.shape[1]
        labels = input_ids.clone()
        labels[:prompt_len] = -100
        labels[attention_mask == 0] = -100

        return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels}


dataset = InstructionDataset(ds, tokenizer, max_length=128)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)
print(f'Dataset: {len(dataset)} examples, {len(dataloader)} batches')

In [ ]:
# ── SFT Training (~2 min on T4, ~20 min on CPU) ──────────────────────────
N_EPOCHS = 3
LR = 2e-5

# Keep a fresh copy for before/after comparison
model_sft = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(DEVICE)
model_sft.train()

optimizer = AdamW(model_sft.parameters(), lr=LR, weight_decay=0.01)
total_steps = N_EPOCHS * len(dataloader)
scheduler = get_scheduler('cosine', optimizer, num_warmup_steps=total_steps//10,
                          num_training_steps=total_steps)

train_losses = []
t0 = time.time()

for epoch in range(N_EPOCHS):
    epoch_loss = 0.0
    for batch in tqdm(dataloader, desc=f'Epoch {epoch+1}/{N_EPOCHS}'):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model_sft(**batch)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_sft.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        epoch_loss += loss.item()
        train_losses.append(loss.item())
    print(f'  Epoch {epoch+1}: avg loss = {epoch_loss/len(dataloader):.4f}')

print(f'\nTraining complete in {time.time()-t0:.1f}s')

In [ ]:
# Plot training loss
fig, ax = plt.subplots(figsize=(9, 4))
smoothed = np.convolve(train_losses, np.ones(5)/5, mode='valid')
ax.plot(train_losses, color='#BCE2E7', alpha=0.5, linewidth=1, label='Raw loss')
ax.plot(range(4, len(train_losses)), smoothed, color='#20808D', linewidth=2, label='Smoothed (window=5)')
for i, ep in enumerate([len(dataloader), 2*len(dataloader)]):
    ax.axvline(ep, color='#A84B2F', linestyle='--', alpha=0.6, label=f'Epoch {i+2}' if i==0 else '')
ax.set_xlabel('Training Step')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('§11.3 — SFT Training Loss Curve (GPT-2 on Dolly-100)', fontweight='bold')
ax.legend()
ax.grid(alpha=0.4)
plt.tight_layout()
plt.savefig('ch11_fig3_sft_loss.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Before / After generation comparison ─────────────────────────────────
model_base = model  # original GPT-2
model_sft.eval()

test_instructions = [
    "What is machine learning?",
    "Explain the concept of overfitting in one sentence.",
]

def generate_response(mdl, instruction, max_new=60):
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    inp = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = mdl.generate(**inp, max_new_tokens=max_new,
                           do_sample=True, temperature=0.7, top_p=0.9,
                           pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)

print('=' * 70)
for inst in test_instructions:
    print(f'\nInstruction: {inst}')
    print(f'[Base GPT-2]:\n  {generate_response(model_base, inst)[:200]}')
    print(f'[SFT GPT-2] :\n  {generate_response(model_sft, inst)[:200]}')
    print('-' * 70)

<div style="background: #1C1B19; border-left: 4px solid #A84B2F; padding: 15px; border-radius: 6px; margin: 15px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">§11.4 Perplexity</h2>
</div>

**Perplexity** measures how surprised the model is by the test data:

\[ \text{PPL}(\mathcal{D}) = \exp\!\left(-\frac{1}{N}\sum_{i=1}^{N} \log P_\theta(x^{(i)})\right) \]

Lower perplexity = the model assigns higher probability to the data = better fit. SFT should decrease perplexity on in-domain instruction data.

In [ ]:
def compute_perplexity(mdl, dataloader, device):
    mdl.eval()
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for batch in dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = mdl(**batch)
            # count only non-masked tokens
            valid_tokens = (batch['labels'] != -100).sum().item()
            total_loss += outputs.loss.item() * valid_tokens
            total_tokens += valid_tokens
    avg_nll = total_loss / total_tokens
    return math.exp(avg_nll)

ppl_base = compute_perplexity(model_base, dataloader, DEVICE)
ppl_sft  = compute_perplexity(model_sft, dataloader, DEVICE)

print(f'Perplexity (Base GPT-2)  : {ppl_base:.2f}')
print(f'Perplexity (SFT GPT-2)   : {ppl_sft:.2f}')
print(f'Reduction                : {100*(1-ppl_sft/ppl_base):.1f}%')

fig, ax = plt.subplots(figsize=(6, 4))
models_names = ['Base GPT-2', 'SFT GPT-2']
ppls = [ppl_base, ppl_sft]
bar_colors = ['#A84B2F', '#20808D']
bars = ax.bar(models_names, ppls, color=bar_colors, width=0.5)
for bar, val in zip(bars, ppls):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.5, f'{val:.1f}',
            ha='center', va='bottom', fontweight='bold', fontsize=12)
ax.set_ylabel('Perplexity (lower = better fit)')
ax.set_title('§11.4 — Perplexity Before vs After SFT', fontweight='bold')
ax.set_ylim(0, max(ppls) * 1.2)
plt.tight_layout()
plt.savefig('ch11_fig4_perplexity.png', dpi=120, bbox_inches='tight')
plt.show()

<div style="background: #1C1B19; border-left: 4px solid #20808D; padding: 15px; border-radius: 6px; margin: 15px 0;">
  <h2 style="color: #BCE2E7; margin: 0;">Summary & Key Takeaways</h2>
</div>

| Concept | Key Result |
|---------|------------|
| Autoregressive generation | GPT-2 samples token-by-token from \(P(x_t \mid x_{<t})\) |
| Temperature | Low \(T\) → sharp/repetitive; high \(T\) → diverse/noisy |
| Top-k / Top-p | Truncate tails for quality without full greediness |
| SFT | Fine-tuning on 100 instruction pairs lowers perplexity and improves instruction-following |
| Perplexity | Exponentiated average NLL — measures model surprise on test data |

**Next:** Chapter 12 covers DPO — fine-tuning from preferences directly without a separate RL training loop.